In [1]:
from langchain_core.tools import tool 
from langchain_aws import ChatBedrockConverse 
from langchain.agents import create_agent 
from langchain.agents.middleware import PIIMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
llm = ChatBedrockConverse(model_id = "cohere.command-r-plus-v1:0", temperature = 0.4, region_name = 'us-east-1')

In [3]:
@tool 
def check_account_balance(account_id: str) -> str: 
    """ Retrieves the current balance for a specific account ID.""" 
    return f"The balance for account {account_id} is $12450.00." 

@tool 
def log_support_ticket(priority: str, category: str) -> str: 
    """Creates a support ticket in the internal CRM.""" 
    return f"Ticket created: [Priority: {priority}] [Category: {category}]." 

tools = [check_account_balance, log_support_ticket]

In [10]:
SYSTEM_PROMPT = """ You are a secure banking assistant. 
Sensitive data in your input is masked for security. Process the request using available tools"""
agent = create_agent(model = llm, tools = tools, system_prompt = SYSTEM_PROMPT, 
                     middleware = [PIIMiddleware("email",strategy="redact", apply_to_input=True), 
                                  PIIMiddleware("ip",strategy = 'mask', apply_to_input = True), 
                                  PIIMiddleware("credit_card", strategy = 'mask', apply_to_input = True)])

In [11]:
def run_secure_session(user_query: str):
    print(f"-----Processing Request -----\nRaw Input: {user_query}\n")
    response = agent.invoke({"messages":user_query})
    print(response['messages']) 
    final_output = response['messages'][-1].content 
    return final_output

In [12]:
user_query = """My email is bac@corp.com. 
Check my balance for account ACC-967 and log a 'High' priority ticket for billing.
Check if this IP address is still active. 127.0.0.1
Check if the this credit card number is still active: 3445-7866-1655-1220""" 
result = run_secure_session(user_query) 
print("-----Agent Final Response------") 
print(result)

-----Processing Request -----
Raw Input: My email is bac@corp.com. 
Check my balance for account ACC-967 and log a 'High' priority ticket for billing.
Check if this IP address is still active. 127.0.0.1
Check if the this credit card number is still active: 3445-7866-1655-1220

[HumanMessage(content="My email is [REDACTED_EMAIL]. \nCheck my balance for account ACC-967 and log a 'High' priority ticket for billing.\nCheck if this IP address is still active. *.*.*.1\nCheck if the this credit card number is still active: 3445-7866-1655-1220", additional_kwargs={}, response_metadata={}, id='66949d81-ed4f-40dd-8503-c7088c2b26ab'), AIMessage(content=[{'type': 'text', 'text': 'I will check the account balance for ACC-967, log a support ticket for billing, and inform the user that I cannot check the IP address or credit card number.'}, {'type': 'tool_use', 'name': 'check_account_balance', 'input': {'account_id': 'ACC-967'}, 'id': 'tooluse_2Bt52tu5VOl2W1tTt7o25Y'}, {'type': 'tool_use', 'name': 'l